# Sprint 3 – Advanced Statistics & Feature Engineering


**Goal:** We will use math to prove our earlier findings, create new helpful data columns, and give final advice to the gaming company.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# Loading the cleaned dataset from Sprint 1
df = pd.read_csv('cleaned_videogame_sales.csv')
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


# Task 1: Statistical Analysis

We will look deeper into the shape and spread of the Global Sales numbers to understand market behavior.

In [2]:
# Advanced Statistics
sales = df['Global_Sales']

# Calculate statistical measures
mean_val = sales.mean()
variance_val = sales.var()
std_val = sales.std()
skewness = sales.skew()
kurt = sales.kurtosis()

print(f"Mean (Average): {mean_val:.2f}")
print(f"Variance (Spread squared): {variance_val:.2f}")
print(f"Standard Deviation (Average spread): {std_val:.2f}")
print(f"Skewness (Leaning to one side): {skewness:.2f}")
print(f"Kurtosis (Tallness/Extremes): {kurt:.2f}")

# Covariance between NA and EU sales
cov_matrix = df[['NA_Sales', 'EU_Sales']].cov()
print("\nCovariance between North America and Europe:")
display(cov_matrix)

Mean (Average): 0.54
Variance (Spread squared): 2.46
Standard Deviation (Average spread): 1.57
Skewness (Leaning to one side): 17.30
Kurtosis (Tallness/Extremes): 596.00

Covariance between North America and Europe:


,NA_Sales,EU_Sales
NA_Sales,0.676395,0.322077
EU_Sales,0.322077,0.259389


### Task 1 Observations: What the Numbers Tell Us

* **High Spread:** The standard deviation is much larger than the average. This means game sales are highly unpredictable. Some games make almost nothing, while others make billions.
* **Extreme Skew and Kurtosis:** A skewness over 10 and a massive kurtosis number mathematically prove that the gaming industry is completely ruled by rare "mega-hits". 
* **Covariance:** The positive covariance between North America and Europe means that as sales go up in America, they also go up in Europe.

## Task 2: Hypothesis Testing

**Research Question:** Do "Action" games sell significantly more copies on average than "Sports" games globally?

* **Null Hypothesis (H0):** There is NO difference in average sales between Action and Sports games.
* **Alternate Hypothesis (HA):** There IS a difference in average sales between Action and Sports games.
* **Significance Level:** 0.05 (We want to be 95% sure).
* **Test Chosen:** Independent T-Test (comparing the averages of two different groups).

In [3]:
# Separate the sales data for Action and Sports games
action_sales = df[df['Genre'] == 'Action']['Global_Sales']
sports_sales = df[df['Genre'] == 'Sports']['Global_Sales']

# Run the Independent T-Test
t_stat, p_value = stats.ttest_ind(action_sales, sports_sales, equal_var=False)

print(f"Test Statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Decision: Reject the Null Hypothesis")
else:
    print("Decision: Fail to reject the Null Hypothesis")

Test Statistic: -0.7917
P-value: 0.4286
Decision: Fail to reject the Null Hypothesis


### Task 2 Observations: Business Interpretation[cite: 10]

* **The Result:** The p-value is greater than 0.05, so we fail to reject the null hypothesis. 
* **Business Meaning:** Even though we saw earlier that Action and Sports look slightly different on charts, mathematically, their average sales are roughly the same. A company should not choose to make an Action game over a Sports game just because they think it will automatically sell more copies on average.

## Task 3: Confidence Interval Analysis

We will calculate a 95% confidence interval for average Global Sales. This gives us a safe estimated range for how much an average game will sell.

In [4]:
# Calculate 95% Confidence Interval for Global Sales
data = df['Global_Sales']
confidence = 0.95
n = len(data)
m = np.mean(data)
std_err = stats.sem(data)

h = std_err * stats.t.ppf((1 + confidence) / 2, n - 1)

start = m - h
end = m + h

print(f"95% Confidence Interval for Average Global Sales: {start:.3f} to {end:.3f} million units")

95% Confidence Interval for Average Global Sales: 0.517 to 0.565 million units


## Task 4: Feature Engineering

Feature engineering means creating new meaningful features from existing data to improve analysis and business understanding.

### New Feature: `Hit_Status`

We will create a `Hit_Status` feature using `Global_Sales`.

### Logic

* If `Global_Sales >= 1.0` million → **Mega-Hit**
* If `Global_Sales < 1.0` million → **Standard**

The 1 million threshold is used as an analytical classification for this project to separate relatively high-selling games from the rest.

### Business Purpose

Average sales alone can hide the difference between ordinary-performing games and highly successful games. The new feature allows us to analyze the proportion of high-selling games across genres, platforms, publishers, and regions.

### Justification

This feature is useful because it converts a continuous sales variable into a meaningful business category that can be used for segmentation and performance comparison.


In [5]:
# Create the Hit_Status feature

df['Hit_Status'] = np.where(
    df['Global_Sales'] >= 1.0,
    'Mega-Hit',
    'Standard'
)

# Check the new feature
display(df[['Name', 'Global_Sales', 'Hit_Status']].head())

,Name,Global_Sales,Hit_Status
0,Wii Sports,82.74,Mega-Hit
1,Super Mario Bros.,40.24,Mega-Hit
2,Mario Kart Wii,35.82,Mega-Hit
3,Wii Sports Resort,33.00,Mega-Hit
4,Pokemon Red/Pokemon Blue,31.37,Mega-Hit


In [6]:
# Count games in each category

hit_counts = df['Hit_Status'].value_counts()

print("Number of Games by Hit Status:")
display(hit_counts)

Number of Games by Hit Status:


Hit_Status
Standard    14233
Mega-Hit     2058
Name: count, dtype: int64

In [7]:
# Percentage of games in each category

hit_percentage = (
    df['Hit_Status']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Percentage of Games by Hit Status:")
display(hit_percentage)

Percentage of Games by Hit Status:


Hit_Status
Standard    87.37
Mega-Hit    12.63
Name: proportion, dtype: float64

Genre Analysis

In [8]:
# Calculate percentage of Mega-Hits within each genre

hit_rates = (
    pd.crosstab(
        df['Genre'],
        df['Hit_Status'],
        normalize='index'
    ) * 100
)

hit_rates = hit_rates.sort_values(
    by='Mega-Hit',
    ascending=False
)

print("Mega-Hit Percentage by Genre:")
display(hit_rates)

Mega-Hit Percentage by Genre:


Hit_Status,Mega-Hit,Standard
Genre,,
Platform,22.171429,77.828571
Shooter,19.422777,80.577223
Racing,14.775510,85.224490
Fighting,14.712919,85.287081
Role-Playing,13.809524,86.190476
Sports,13.020833,86.979167
Action,12.888342,87.111658
Simulation,10.966981,89.033019
Misc,9.964413,90.035587


this tells - What percentage of games in each genre reach the Mega-Hit category?

# TASK 5 — Feature Evaluation

## Task 5: Feature Evaluation



### Before Feature Engineering

Before creating `Hit_Status`, the analysis mainly focused on numerical sales measures such as:

* Total Global Sales
* Average Global Sales
* Regional Sales

These measures describe sales performance but do not directly show how frequently games reach a high-sales category.

### After Feature Engineering

The `Hit_Status` feature allows games to be classified into `Mega-Hit` and `Standard` categories.

This makes it possible to compare the percentage of Mega-Hits across genres and identify which genres have a higher proportion of high-selling games.

### Evaluation

If the `Hit_Status` analysis reveals meaningful differences between genres, platforms, or publishers, then the engineered feature provides additional business information that was not visible from average sales alone.

Therefore, the feature improves segmentation and makes high-performing games easier to identify.


In [12]:
# BEFORE feature engineering
average_genre_sales = (
    df.groupby('Genre')['Global_Sales']
    .mean()
    .sort_values(ascending=False)
)

print("Before Feature Engineering - Average Sales by Genre:")
display(average_genre_sales)
#
#
#
#
print("-----------------------------------------------------------------------")
# AFTER feature engineering
print("After Feature Engineering - Mega-Hit Rate by Genre:")
display(hit_rates[['Mega-Hit']].sort_values(
    by='Mega-Hit',
    ascending=False
))




Before Feature Engineering - Average Sales by Genre:


Genre
Platform        0.947577
Shooter         0.800468
Role-Playing    0.628456
Racing          0.593273
Sports          0.568247
Fighting        0.531160
Action          0.529942
Misc            0.468488
Simulation      0.459882
Puzzle          0.424930
Strategy        0.258612
Adventure       0.184137
Name: Global_Sales, dtype: float64

-----------------------------------------------------------------------
After Feature Engineering - Mega-Hit Rate by Genre:


Hit_Status,Mega-Hit
Genre,
Platform,22.171429
Shooter,19.422777
Racing,14.775510
Fighting,14.712919
Role-Playing,13.809524
Sports,13.020833
Action,12.888342
Simulation,10.966981
Misc,9.964413


### Task 5: Feature Evaluation — Final Interpretation

**Before:**
Average Global Sales showed which genres generated higher sales on average.

**After:**
`Hit_Status` provided an additional perspective by showing the percentage of games that reached the defined Mega-Hit threshold.

**What improved?**
The new feature makes it easier to compare the likelihood of high-sales performance across different genres.

**Business Value:**
Publishers and developers can use both average sales and Mega-Hit rates when evaluating opportunities. Average sales show typical performance, while Mega-Hit rate shows how frequently games reach the project's high-sales threshold.

**Conclusion:**
The `Hit_Status` feature adds meaningful segmentation to the dataset rather than simply increasing the number of columns. It provides additional information that was not directly available from the original continuous sales variable.


# Final Business Recommendations

### For Game Publishers

* **Plan for realistic sales, while targeting high-potential titles:** The analysis shows that game sales are highly uneven, with most games generating moderate sales and a small number achieving very high sales. Publishers should build financial plans around realistic expected sales rather than assuming every title will become a blockbuster. At the same time, they should identify opportunities for high-potential titles that could become Mega-Hits.

### For Game Developers

* **Choose genres based on both sales performance and hit potential:** The analysis shows differences in sales performance and Mega-Hit rates across genres. Developers aiming for higher commercial potential should prioritize genres that demonstrate stronger historical performance in the dataset, such as Shooters and Platform games. However, genre selection should also consider competition, target audience, platform, and development costs rather than guaranteeing success based on genre alone.

### For Marketing Teams

* **Use regional marketing strategies:** North America and Europe show strong relationships with overall game sales, indicating that performance in these markets is closely associated with global performance. Marketing teams can use similar campaign themes where consumer behavior is comparable, while still adapting messaging and budget allocation to each market.

* **Develop a differentiated strategy for Japan:** Japan shows different sales patterns from Western markets, particularly in genre preferences. Marketing teams should therefore avoid applying the same strategy across all regions and should consider localized campaigns that emphasize genres and game characteristics with stronger demand in the Japanese market.

### For Platform Owners — Sony, Microsoft, and Nintendo

* **Use platform-specific strengths when building game portfolios:** The analysis shows that sales performance varies across platforms and genres. Platform owners should use these historical patterns when deciding which games, genres, partnerships, and exclusive content to prioritize.

* **Align game portfolios with their strongest audiences:** Rather than assuming that one genre will perform equally well on every platform, platform owners should analyze the combination of **platform + genre + regional demand** before investing in major partnerships or exclusive releases.

### Overall Recommendation

The analysis suggests that gaming businesses should avoid relying on a single metric when making strategic decisions. **Genre, platform, publisher, region, sales distribution, time trends, and Mega-Hit rates should be considered together** to identify opportunities and manage risk.

The key principle is to use historical data to guide decisions while recognizing that **correlation and past performance do not guarantee future success**.
